### ***RAG System (Retrieval-Augmented Generation)***
- ***It is an AI architectural pattern that optimizes the output of a Large Language Model (LLM) by referencing an authoritative external knowledge base outside its training data before generating a response.***

In [1]:
import warnings 
warnings.filterwarnings("ignore")

from langchain_core.documents import Document
from langchain_community.document_loaders.text import TextLoader 
loader = TextLoader("../data/blog.txt",encoding="utf-8")
docs = loader.load()
print(docs)

[Document(metadata={'source': '../data/blog.txt'}, page_content='Building Smarter AI Applications: A Guide to Retrieval-Augmented Generation (RAG) with Python\n\n1. Introduction\nLarge Language Models (LLMs) are powerful, but they share common limitations: knowledge cutoffs and hallucinations. Training or fine-tuning models on private or rapidly changing data is often slow and expensive. \n\nRetrieval-Augmented Generation (RAG) solves this by pairing an external information retrieval mechanism with a generative model. Instead of relying solely on internal parameters, the model searches a curated knowledge base, pulls relevant context, and uses it to synthesize accurate, verifiable responses.\n\n2. Why Python Powers Modern RAG\nPython is the industry standard for AI and machine learning workflows due to its robust ecosystem:\n- AI Orchestration: Frameworks like LangChain and LlamaIndex provide high-level abstractions to construct pipelines quickly.\n- Vector Search & Embeddings: Native 

In [2]:
# Load PDF Docs 
from langchain_community.document_loaders.pdf import PyPDFLoader
pdf_loader = PyPDFLoader("../data/rag.pdf")
docs = pdf_loader.load()
len(docs)

21

#### ***Ingestion Pipeline***

- ***Pipeline that collects unstructured data (PDFs, Markdown, databases), cleans it, and splits it into manageable pieces (chunking).***

In [3]:
import os

def load_data():
     folder_path = "../data"
     num_docs = 0 
     all_docs = []

     for filename in os.listdir(folder_path):
          if filename.lower().endswith(".pdf"):
               pdf_path = os.path.join(folder_path,filename)
               loader = PyPDFLoader(pdf_path)
               docs = loader.load()
               all_docs.extend(docs)
               num_docs+=1
     print(f"Total PDF:{num_docs}")
     print(f"Total Pages:{len(all_docs)}")
     return all_docs

In [4]:
pdfs = load_data()

Total PDF:2
Total Pages:32


In [5]:
type(pdfs[1])

langchain_core.documents.base.Document

### ***Chunking***
- ***Chunking is the process of splitting large documents into smaller, semantically coherent text segments with controlled token limits and overlap for optimal vector embedding and retrieval.***

In [6]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

def split_docs(docs):
     text_splitter = RecursiveCharacterTextSplitter(
          chunk_size=500,
          chunk_overlap=89
     )
     chunked_docs = text_splitter.split_documents(docs)
     return chunked_docs

In [7]:
chunks = split_docs(pdfs)
len(chunks)

352

### ***Embedding***
- ***Converts chunked text into dense vector representations capturing semantic meaning (e.g., text-embedding-3-small, BAAI/bge-large).***

In [8]:
from sentence_transformers import SentenceTransformer

# Class Manager 
class EmbeddingManager:
    def __init__(self, model_name="all-MiniLM-L6-v2"):
        
        self.model_name=model_name
        print("loading model...🔃", self.model_name)
        self.model = SentenceTransformer(self.model_name)
        print("embedding dimensions=", self.model.get_sentence_embedding_dimension())


    def generate_embeddings(self, text):
        embeddings = self.model.encode(text, show_progress_bar=True)
        print("embeddings shape:", embeddings.shape)
        return embeddings

In [9]:
os.getenv("HF_TOKEN")
embedding_manager = EmbeddingManager()

loading model...🔃 all-MiniLM-L6-v2


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

embedding dimensions= 384


### ***Vector Store***
- ***Indexes and stores embeddings for rapid high-dimensional similarity searches (e.g., Pinecone, Milvus, Chroma, pgvector).***

In [10]:
import chromadb 
import uuid 

class VectorStoreManager:
    def __init__(self, persist_directory="../data/vector_store", collection_name="pdf_documents"):
        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.collection = None
        self.client = None
        self._initialize_store()

    def _initialize_store(self):
        os.makedirs(self.persist_directory, exist_ok=True)
        
        # create a client
        self.client = chromadb.PersistentClient(path=self.persist_directory)

        # create the collection
        self.collection = self.client.get_or_create_collection(
            name=self.collection_name,
            metadata={"description": "vector store collection for pdf embeddings in RAG"}
        )

        print("initialized the vector store with collection success✅", self.collection_name)
        print("docs in collection:", self.collection.count())

    def add_documents(self, documents, embeddings):
        if len(documents) != len(embeddings):
            raise ValueError("num of documents does not match num of embeddings")


        # store => ids, embedding, document, metadata
        ids = []
        all_metadata = []
        documents_content = []
        embeddings_list = []

        for i, (doc, embedding) in enumerate(zip(documents, embeddings)):
          doc_id = f"doc_{uuid.uuid4()}"
          ids.append(doc_id)

          metadata = dict(doc.metadata)
          metadata["doc_index"] = i
          metadata["content_length"] = len(doc.page_content)
          all_metadata.append(metadata)

          documents_content.append(doc.page_content)

          embeddings_list.append(embedding.tolist())

          self.collection.add(
               ids=ids,
               metadatas=all_metadata,
               documents=documents_content,
               embeddings=embeddings_list
          )

        print("total documents added in vector store:", len(documents_content))
        print("docs in collection:", self.collection.count())

In [11]:
vector_store = VectorStoreManager()

initialized the vector store with collection success✅ pdf_documents
docs in collection: 352


In [12]:
# data => documents => chunks => embeddings => store in vector store

texts = [doc.page_content for doc in chunks]
emebedding = embedding_manager.generate_embeddings(texts)
vector_store.add_documents(chunks, emebedding)

Batches:   0%|          | 0/11 [00:00<?, ?it/s]

embeddings shape: (352, 384)
total documents added in vector store: 352
docs in collection: 704


### ***Retrieval Pipeline***
- ***Finds the most relevant chunks given a user query by calculating distance metrics like Cosine Similarity, Dot Product, or using hybrid search (BM25 keyword search + vector search).***

In [13]:
from sklearn.metrics.pairwise import cosine_similarity

# Retriever Class
class RAGRetriever:
    def __init__(self, embedding_manager, vector_store):
        self.embedding_manager = embedding_manager
        self.vector_store = vector_store


    def retrieve(self, query, top_k=5, score_threshold=0.0):
        # query => embedding
        query_embeddings = self.embedding_manager.generate_embeddings([query])[0]

        # semantic search
        results = self.vector_store.collection.query(
            query_embeddings=[query_embeddings.tolist()],
            n_results=top_k
        )

        # cosine similarity
        retrieved_docs=[]
        
        if results["documents"] and results["documents"][0]:
            ids = results["ids"][0]
            metadatas = results["metadatas"][0]
            documents = results["documents"][0]
            distances = results["distances"][0]

            for i, (doc_id, metadata, document, distance) in enumerate(zip(ids, metadatas, documents, distances)):
                similarity_score = 1 - distance

                if similarity_score >= score_threshold:
                    retrieved_docs.append({
                        "id": doc_id,
                        "document": document,
                        "metadata": metadata,
                        "distance": distance,
                        "similarity_score": similarity_score,
                        "rank" : i + 1
                    })

            print(f"retrieved {len(retrieved_docs)} documents")

        else:
            print("no documents found")
        return f"{retrieved_docs} success..✅"

In [14]:
rag_retriever = RAGRetriever(embedding_manager, vector_store)

In [15]:
answer = rag_retriever.retrieve("What is rag")
print(answer)

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

embeddings shape: (1, 384)
retrieved 5 documents
[{'id': 'doc_242951da-80c9-48c0-8437-2ed708739f06', 'document': 'to generate a well-informed answer.\nThe RAG research paradigm is continuously evolving, and\nwe categorize it into three stages: Naive RAG, Advanced\nRAG, and Modular RAG, as showed in Figure 3. Despite\nRAG method are cost-effective and surpass the performance\nof the native LLM, they also exhibit several limitations.\nThe development of Advanced RAG and Modular RAG is\na response to these specific shortcomings in Naive RAG.\nA. Naive RAG\nThe Naive RAG research paradigm represents the earli-', 'metadata': {'producer': 'pdfTeX-1.40.25', 'total_pages': 21, 'author': '', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.25 (TeX Live 2023) kpathsea version 6.3.5', 'title': '', 'moddate': '2024-03-28T00:54:45+00:00', 'doc_index': 19, 'keywords': '', 'trapped': '/False', 'source': '../data\\rag.pdf', 'creationdate': '2024-03-28T00:54:45+00:00', 'page_label': '2'

In [16]:
answer = rag_retriever.retrieve("What is encoder and decoder stacks")
print(answer)

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

embeddings shape: (1, 384)
retrieved 5 documents
[{'id': 'doc_2b6fc7f9-a5d3-45ff-8128-049c418ef45d', 'document': 'respectively.\n3.1 Encoder and Decoder Stacks\nEncoder: The encoder is composed of a stack of N = 6 identical layers. Each layer has two\nsub-layers. The ﬁrst is a multi-head self-attention mechanism, and the second is a simple, position-\n2\nankig4515@gmail.com', 'metadata': {'subject': 'Neural Information Processing Systems http://nips.cc/', 'published': '2017', 'creator': 'PyPDF', 'eventtype': 'Poster', 'firstpage': '5998', 'producer': 'pdfcpu v0.12.1 dev', 'lastpage': '6008', 'description': 'Paper accepted and presented at the Neural Information Processing Systems Conference (http://nips.cc/)', 'language': 'en-US', 'book': 'Advances in Neural Information Processing Systems 30', 'title': 'Attention is All you Need', 'doc_index': 289, 'author': 'Ashish Vaswani, Noam Shazeer, Niki Parmar, Jakob Uszkoreit, Llion Jones, Aidan N. Gomez, Łukasz Kaiser, Illia Polosukhin', 'date

### ***Integrate LLM***
- ***Groq*** 
- ***MistralAI***
- ***Openai(paid)***

In [17]:
import os 
from dotenv import load_dotenv
load_dotenv()

True

In [34]:
from langchain_groq import ChatGroq 
api = os.getenv("GROQ_API_KEY")

llm = ChatGroq(
     api_key=api,
     model="openai/gpt-oss-120b",
     temperature=0.3,
     streaming=True
)

In [35]:
# Genarate Output 
def genrate_output(query, retriever, llm, top_k=3):
    results = retriever.retrieve(query, top_k)
    context = "\n".join(results) if results else ""

    if not context:
        print("not found relevant information ❌")
        return None

    prompt = f"""use given context to generate the answer for the query
               Context: {context}
               Query: {query}
             """

    response = llm.invoke([prompt])
    return response.content

In [36]:
answer = genrate_output("what is RAG",rag_retriever,llm)

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

embeddings shape: (1, 384)
retrieved 3 documents


In [37]:
print(answer)

**RAG (Retrieval‑Augmented Generation)** is a research paradigm that couples a language model’s generative abilities with an external knowledge‑retrieval component.  
Instead of relying solely on the model’s internal parameters, RAG first fetches relevant documents (or other pieces of information) from a datastore and then conditions the generation on that retrieved content.  

The literature you provided describes how the RAG paradigm is evolving through three stages:

1. **Naïve RAG** – the earliest form, where a basic retrieval step is added to a language model.  
2. **Advanced RAG (nRAG)** – more sophisticated methods that improve cost‑effectiveness and often outperform the native LLM, though they still have notable limitations.  
3. **Modular RAG** – a newer, more flexible architecture designed to address the shortcomings of the naïve approach.

In short, RAG augments a generative model with up‑to‑date external information, aiming to produce more accurate, factual, and context‑awa